In [10]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import os
import hashlib
import base58
from bitcoinlib.keys import Key
import random
from IPython.display import display
import ipywidgets as widgets
from collections import defaultdict
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# Function to calculate Hamming distance (bits off) between two byte strings
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF starting with '5' (uncompressed) with optional prefix
def generate_wif_starting_with_5(prefix='5'):
    while True:
        private_key_bytes = os.urandom(32)
        key = Key(private_key_bytes)
        extended_key = b'\x80' + private_key_bytes
        checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
        wif = base58.b58encode(extended_key + checksum).decode('utf-8')
        if wif.startswith(prefix):
            return wif, key

# Function to decode WIF and derive hash160 and address (uncompressed)
def derive_address_and_hash160(wif):
    decoded = base58.b58decode(wif)
    private_key_bytes = decoded[1:-4]
    sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
    vk = sk.verifying_key
    public_key = b'\x04' + vk.to_string()
    sha256_hash = hashlib.sha256(public_key).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(sha256_hash)
    hash160 = ripemd160.digest()
    version = b'\x00'
    data = version + hash160
    checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
    address = base58.b58encode(data + checksum).decode('utf-8')
    return address, hash160

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Main intelligent analysis function
def analyze_wif_patterns(num_attempts_per_round=100, max_rounds=5):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)',
        description='Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            base58_chars = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'
            current_prefix = '5'
            position = 1

            for round_num in range(max_rounds):
                print(f"\nRound {round_num + 1}: Analyzing prefix '{current_prefix}'")
                # Use defaultdict to track sum and count for each character
                char_stats = defaultdict(lambda: {'sum': 0, 'count': 0})
                start_time = time.time()

                for attempt in range(num_attempts_per_round):
                    wif, _ = generate_wif_starting_with_5(prefix=current_prefix)
                    _, gen_hash160 = derive_address_and_hash160(wif)
                    distance = hamming_distance(gen_hash160, target_hash160)
                    next_char = wif[position] if len(wif) > position else None
                    if next_char:
                        char_stats[next_char]['sum'] += distance
                        char_stats[next_char]['count'] += 1

                    # Timeout check
                    if time.time() - start_time > 10:  # 10-second timeout per round
                        print(f"Round {round_num + 1} timed out after 10 seconds.")
                        break

                if not char_stats:
                    print("No viable WIFs generated. Try increasing attempts.")
                    break

                # Calculate average distance per character
                avg_distances = {char: stats['sum'] / stats['count']
                               for char, stats in char_stats.items() if stats['count'] > 0}

                if not avg_distances:
                    print("No valid distance data collected.")
                    break

                # Find the best character
                best_char = min(avg_distances, key=avg_distances.get)
                min_distance = avg_distances[best_char]

                print(f"Position {position + 1} candidates (avg bits off):")
                for char in sorted(avg_distances.keys())[:5]:
                    print(f"  '{char}': {avg_distances[char]:.2f} (count: {char_stats[char]['count']})")
                print(f"Best next char: '{best_char}' (avg {min_distance:.2f} bits off)")

                # Update prefix
                current_prefix += best_char
                position += 1

                # Example WIF with new prefix
                example_wif, _ = generate_wif_starting_with_5(current_prefix)
                example_addr, example_hash160 = derive_address_and_hash160(example_wif)
                print(f"Example WIF: {example_wif}")
                print(f"Example Address: {example_addr}")
                print(f"Example Hash160: {example_hash160.hex()}")

                # Check for match
                if example_hash160 == target_hash160:
                    print(f"\nSuccess! Found matching WIF: {example_wif}")
                    break

                if position >= 51:
                    print("Reached maximum WIF length without a match.")
                    break

                # Progress check
                elapsed = time.time() - start_time
                print(f"Round {round_num + 1} completed in {elapsed:.2f} seconds")

            print("\n--- Analysis Complete ---")
            print(f"Final prefix explored: '{current_prefix}'")
            print("Adjust num_attempts_per_round or max_rounds for deeper analysis.")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(num_attempts_per_round=100, max_rounds=5)

1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF


Text(value='', description='Address:', layout=Layout(width='500px'), placeholder='Enter your Bitcoin address (…

Button(description='Submit', style=ButtonStyle())

Output()

In [12]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import os
import hashlib
import base58
from bitcoinlib.keys import Key
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key with '5' prefix
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
    vk = sk.verifying_key
    public_key = b'\x04' + vk.to_string()  # Uncompressed
    sha256_hash = hashlib.sha256(public_key).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(sha256_hash)
    hash160 = ripemd160.digest()
    version = b'\x00'
    data = version + hash160
    checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
    address = base58.b58encode(data + checksum).decode('utf-8')
    return address, hash160

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Main analysis function with hill-climbing logic
def analyze_wif_patterns(max_rounds=5, attempts_per_position=50):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)',
        description='Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with a random 32-byte private key
            base_private_key = bytearray(os.urandom(32))
            best_distance = float('inf')
            best_key = base_private_key[:]
            position = 0  # Byte position to tweak

            for round_num in range(max_rounds):
                print(f"\nRound {round_num + 1}: Tweaking byte position {position}")
                start_time = time.time()

                # Try modifying the current byte position
                current_key = best_key[:]
                current_wif = private_key_to_wif(current_key)
                current_addr, current_hash160 = derive_address_and_hash160(current_key)
                current_distance = hamming_distance(current_hash160, target_hash160)

                print(f"Starting WIF: {current_wif}")
                print(f"Starting Address: {current_addr}")
                print(f"Starting Hash160: {current_hash160.hex()}")
                print(f"Starting Distance: {current_distance} bits off")

                for attempt in range(attempts_per_position):
                    # Tweak the byte at the current position
                    new_key = current_key[:]
                    new_key[position] = random.randint(0, 255)
                    new_wif = private_key_to_wif(new_key)
                    new_addr, new_hash160 = derive_address_and_hash160(new_key)
                    new_distance = hamming_distance(new_hash160, target_hash160)

                    # Update if better
                    if new_distance < current_distance:
                        current_key = new_key[:]
                        current_wif = new_wif
                        current_addr = new_addr
                        current_hash160 = new_hash160
                        current_distance = new_distance
                        print(f"Improved! Distance: {current_distance} bits off")
                        print(f"New WIF: {current_wif}")
                        print(f"New Address: {current_addr}")
                        print(f"New Hash160: {current_hash160.hex()}")

                    if current_distance == 0:
                        print(f"\nSuccess! Found matching WIF: {current_wif}")
                        return

                # Update best if improved
                if current_distance < best_distance:
                    best_distance = current_distance
                    best_key = current_key[:]

                # Move to next byte position
                position = (position + 1) % 32  # Cycle through 32 bytes
                elapsed = time.time() - start_time
                print(f"Round {round_num + 1} completed in {elapsed:.2f} seconds")
                print(f"Best Distance so far: {best_distance} bits off")

            final_wif = private_key_to_wif(best_key)
            final_addr, final_hash160 = derive_address_and_hash160(best_key)
            print("\n--- Analysis Complete ---")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {best_distance} bits off")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(max_rounds=5, attempts_per_position=50)

Text(value='', description='Address:', layout=Layout(width='500px'), placeholder='Enter your Bitcoin address (…

Button(description='Submit', style=ButtonStyle())

Output()

In [13]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import os
import hashlib
import base58
from bitcoinlib.keys import Key
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
    vk = sk.verifying_key
    public_key = b'\x04' + vk.to_string()  # Uncompressed
    sha256_hash = hashlib.sha256(public_key).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(sha256_hash)
    hash160 = ripemd160.digest()
    version = b'\x00'
    data = version + hash160
    checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
    address = base58.b58encode(data + checksum).decode('utf-8')
    return address, hash160

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Main analysis function with improved hill-climbing
def analyze_wif_patterns(max_rounds=5):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)',
        description='Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with a random 32-byte private key
            best_key = bytearray(os.urandom(32))
            best_distance = float('inf')
            position = 0  # Byte position to tweak

            for round_num in range(max_rounds):
                print(f"\nRound {round_num + 1}: Tweaking byte position {position}")
                start_time = time.time()

                # Current best key
                current_key = best_key[:]
                current_wif = private_key_to_wif(current_key)
                current_addr, current_hash160 = derive_address_and_hash160(current_key)
                current_distance = hamming_distance(current_hash160, target_hash160)

                print(f"Starting WIF: {current_wif}")
                print(f"Starting Address: {current_addr}")
                print(f"Starting Hash160: {current_hash160.hex()}")
                print(f"Starting Distance: {current_distance} bits off")

                # Exhaustively test all 256 values for the current byte
                improved = False
                for value in range(256):
                    test_key = current_key[:]
                    test_key[position] = value
                    test_hash160 = derive_address_and_hash160(test_key)[1]
                    test_distance = hamming_distance(test_hash160, target_hash160)

                    if test_distance < current_distance:
                        current_key = test_key[:]
                        current_wif = private_key_to_wif(current_key)
                        current_addr, current_hash160 = derive_address_and_hash160(current_key)
                        current_distance = test_distance
                        improved = True
                        print(f"Improved! Distance: {current_distance} bits off")
                        print(f"New WIF: {current_wif}")
                        print(f"New Address: {current_addr}")
                        print(f"New Hash160: {current_hash160.hex()}")

                    if current_distance == 0:
                        print(f"\nSuccess! Found matching WIF: {current_wif}")
                        return

                # Update best key
                if improved:
                    best_key = current_key[:]
                    best_distance = current_distance

                # Move to next position, revisit earlier if no improvement
                position = (position + 1) % 32 if improved else (position - 1) % 32
                elapsed = time.time() - start_time
                print(f"Round {round_num + 1} completed in {elapsed:.2f} seconds")
                print(f"Best Distance so far: {best_distance} bits off")

            final_wif = private_key_to_wif(best_key)
            final_addr, final_hash160 = derive_address_and_hash160(best_key)
            print("\n--- Analysis Complete ---")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {best_distance} bits off")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(max_rounds=5)

Text(value='', description='Address:', layout=Layout(width='500px'), placeholder='Enter your Bitcoin address (…

Button(description='Submit', style=ButtonStyle())

Output()

In [14]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import hashlib
import base58
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
    vk = sk.verifying_key
    public_key = b'\x04' + vk.to_string()  # Uncompressed
    sha256_hash = hashlib.sha256(public_key).digest()
    ripemd160 = RIPEMD160.new()
    ripemd160.update(sha256_hash)
    hash160 = ripemd160.digest()
    version = b'\x00'
    data = version + hash160
    checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
    address = base58.b58encode(data + checksum).decode('utf-8')
    return address, hash160

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Main analysis function: Optimize each byte independently
def analyze_wif_patterns():
    address_input = widgets.Text(
        value='',
        placeholder='Enter your Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)',
        description='Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with all zeros
            base_key = bytearray([0] * 32)
            best_bytes = [0] * 32  # Store best value for each position
            best_distances = [float('inf')] * 32  # Store best distance for each position

            start_time = time.time()
            print("\nOptimizing each byte position independently...")

            # Test each byte position
            for position in range(32):
                print(f"\nPosition {position}:")
                for value in range(256):
                    test_key = base_key[:]
                    test_key[position] = value
                    test_hash160 = derive_address_and_hash160(test_key)[1]
                    distance = hamming_distance(test_hash160, target_hash160)

                    if distance < best_distances[position]:
                        best_distances[position] = distance
                        best_bytes[position] = value
                        print(f"  Value {hex(value)}: {distance} bits off (new best)")

            # Combine best bytes into final key
            final_key = bytearray(best_bytes)
            final_wif = private_key_to_wif(final_key)
            final_addr, final_hash160 = derive_address_and_hash160(final_key)
            final_distance = hamming_distance(final_hash160, target_hash160)

            elapsed = time.time() - start_time
            print("\n--- Analysis Complete ---")
            print(f"Combined Key (hex): {final_key.hex()}")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {final_distance} bits off")
            print(f"Time taken: {elapsed:.2f} seconds")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns()

Text(value='', description='Address:', layout=Layout(width='500px'), placeholder='Enter your Bitcoin address (…

Button(description='Submit', style=ButtonStyle())

Output()

In [16]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import hashlib
import base58
import os
import random
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# SECP256k1 curve order
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    try:
        sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
        vk = sk.verifying_key
        public_key = b'\x04' + vk.to_string()  # Uncompressed
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        hash160 = ripemd160.digest()
        version = b'\x00'
        data = version + hash160
        checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
        address = base58.b58encode(data + checksum).decode('utf-8')
        return address, hash160
    except Exception:
        return None, None

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Adaptive search function
def analyze_wif_patterns(max_iterations=1000, tweak_size=2):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)',
        description='Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with a valid random private key
            current_key = bytearray(os.urandom(32))
            current_wif = private_key_to_wif(current_key)
            current_addr, current_hash160 = derive_address_and_hash160(current_key)
            current_distance = hamming_distance(current_hash160, target_hash160)
            best_key = current_key[:]
            best_distance = current_distance

            print(f"Starting WIF: {current_wif}")
            print(f"Starting Address: {current_addr}")
            print(f"Starting Distance: {current_distance} bits off")

            start_time = time.time()
            temperature = 10.0  # Initial "temperature" for simulated annealing

            for iteration in range(max_iterations):
                # Tweak 1 to tweak_size bytes
                new_key = bytearray(current_key)
                num_tweaks = random.randint(1, tweak_size)
                positions = random.sample(range(32), num_tweaks)
                for pos in positions:
                    new_key[pos] = random.randint(0, 255)

                new_wif = private_key_to_wif(new_key)
                new_addr, new_hash160 = derive_address_and_hash160(new_key)
                if new_hash160 is None:
                    continue
                new_distance = hamming_distance(new_hash160, target_hash160)

                # Accept if better or with probability (simulated annealing)
                delta = new_distance - current_distance
                if delta < 0 or random.random() < 2.71828 ** (-delta / temperature):
                    current_key = new_key[:]
                    current_wif = new_wif
                    current_addr = new_addr
                    current_hash160 = new_hash160
                    current_distance = new_distance
                    print(f"Iteration {iteration + 1}: Distance improved to {current_distance} bits off")
                    print(f"New WIF: {current_wif}")
                    print(f"New Address: {current_addr}")
                    print(f"New Hash160: {current_hash160.hex()}")

                # Update best if improved
                if current_distance < best_distance:
                    best_key = current_key[:]
                    best_distance = current_distance
                    print(f"New best distance: {best_distance} bits off")

                # Check for exact match
                if current_distance == 0 or current_addr == target_address:
                    print(f"\nSuccess! Found matching WIF: {current_wif}")
                    break

                # Cool down temperature
                temperature *= 0.995

                # Progress update
                if (iteration + 1) % 100 == 0:
                    elapsed = time.time() - start_time
                    print(f"Iteration {iteration + 1}: Current Distance: {current_distance} bits off, Time: {elapsed:.2f}s")

            final_wif = private_key_to_wif(best_key)
            final_addr, final_hash160 = derive_address_and_hash160(best_key)
            elapsed = time.time() - start_time
            print("\n--- Analysis Complete ---")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {best_distance} bits off")
            print(f"Time taken: {elapsed:.2f} seconds")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(max_iterations=1000, tweak_size=2)

Text(value='', description='Address:', layout=Layout(width='500px'), placeholder='Enter your Bitcoin address (…

Button(description='Submit', style=ButtonStyle())

Output()

In [17]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import hashlib
import base58
import os
import random
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# SECP256k1 curve order
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    try:
        sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
        vk = sk.verifying_key
        public_key = b'\x04' + vk.to_string()  # Uncompressed
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        hash160 = ripemd160.digest()
        version = b'\x00'
        data = version + hash160
        checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
        address = base58.b58encode(data + checksum).decode('utf-8')
        return address, hash160
    except Exception:
        return None, None

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Adaptive search function
def analyze_wif_patterns(max_iterations=10000, tweak_size=2):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your Bitcoin address (e.g., 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa)',
        description='Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with a valid random private key
            current_key = bytearray(os.urandom(32))
            current_wif = private_key_to_wif(current_key)
            current_addr, current_hash160 = derive_address_and_hash160(current_key)
            current_distance = hamming_distance(current_hash160, target_hash160)
            best_key = current_key[:]
            best_distance = current_distance

            print(f"Starting WIF: {current_wif}")
            print(f"Starting Address: {current_addr}")
            print(f"Starting Distance: {current_distance} bits off")

            start_time = time.time()
            temperature = 20.0  # Higher initial temperature for broader exploration

            for iteration in range(max_iterations):
                # Tweak 1 to tweak_size bytes
                new_key = bytearray(current_key)
                num_tweaks = random.randint(1, tweak_size)
                positions = random.sample(range(32), num_tweaks)
                for pos in positions:
                    new_key[pos] = random.randint(0, 255)

                new_wif = private_key_to_wif(new_key)
                new_addr, new_hash160 = derive_address_and_hash160(new_key)
                if new_hash160 is None:
                    continue
                new_distance = hamming_distance(new_hash160, target_hash160)

                # Accept if better or with probability
                delta = new_distance - current_distance
                if delta < 0 or random.random() < 2.71828 ** (-delta / temperature):
                    current_key = new_key[:]
                    current_wif = new_wif
                    current_addr = new_addr
                    current_hash160 = new_hash160
                    current_distance = new_distance
                    print(f"Iteration {iteration + 1}: Distance improved to {current_distance} bits off")
                    print(f"New WIF: {current_wif}")
                    print(f"New Address: {current_addr}")
                    print(f"New Hash160: {current_hash160.hex()}")

                # Update best if improved
                if current_distance < best_distance:
                    best_key = current_key[:]
                    best_distance = current_distance
                    print(f"New best distance: {best_distance} bits off")

                # Check for exact match
                if current_distance == 0 or current_addr == target_address:
                    print(f"\nSuccess! Found matching WIF: {current_wif}")
                    break

                # Cool down temperature
                temperature *= 0.999

                # Progress update
                if (iteration + 1) % 1000 == 0:
                    elapsed = time.time() - start_time
                    print(f"Iteration {iteration + 1}: Current Distance: {current_distance} bits off, Time: {elapsed:.2f}s")

            final_wif = private_key_to_wif(best_key)
            final_addr, final_hash160 = derive_address_and_hash160(best_key)
            elapsed = time.time() - start_time
            print("\n--- Analysis Complete ---")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {best_distance} bits off")
            print(f"Time taken: {elapsed:.2f} seconds")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(max_iterations=10000, tweak_size=2)

Text(value='', description='Address:', layout=Layout(width='500px'), placeholder='Enter your Bitcoin address (…

Button(description='Submit', style=ButtonStyle())

Output()

In [20]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import hashlib
import base58
import os
import random
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# SECP256k1 curve order
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    try:
        sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
        vk = sk.verifying_key
        public_key = b'\x04' + vk.to_string()  # Uncompressed
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        hash160 = ripemd160.digest()
        version = b'\x00'
        data = version + hash160
        checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
        address = base58.b58encode(data + checksum).decode('utf-8')
        return address, hash160
    except Exception:
        return None, None

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Adaptive search function prioritizing address match
def analyze_wif_patterns(max_iterations=100000, tweak_size=3):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your target Bitcoin address',
        description='Target Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with the best key from your last run
            best_wif = "5JxEvR5Z8veRQjBUncvVMtx9pnKz7d26BxEiXZpqHsryF6nsWvS"
            best_key_decoded = base58.b58decode(best_wif)[1:-4]  # Strip version and checksum
            current_key = bytearray(best_key_decoded)
            current_wif = private_key_to_wif(current_key)
            current_addr, current_hash160 = derive_address_and_hash160(current_key)
            current_distance = hamming_distance(current_hash160, target_hash160)
            best_key = current_key[:]
            best_distance = current_distance

            print(f"Starting WIF: {current_wif}")
            print(f"Starting Address: {current_addr}")
            print(f"Starting Distance: {current_distance} bits off")

            start_time = time.time()
            temperature = 30.0

            for iteration in range(max_iterations):
                # Tweak 1 to tweak_size bytes
                new_key = bytearray(current_key)
                num_tweaks = random.randint(1, tweak_size)
                positions = random.sample(range(32), num_tweaks)
                for pos in positions:
                    new_key[pos] = random.randint(0, 255)

                new_wif = private_key_to_wif(new_key)
                new_addr, new_hash160 = derive_address_and_hash160(new_key)
                if new_hash160 is None:
                    continue
                new_distance = hamming_distance(new_hash160, target_hash160)

                # Prioritize address match, then distance
                if new_addr == target_address:
                    best_key = new_key[:]
                    best_wif = new_wif
                    best_addr = new_addr
                    best_hash160 = new_hash160
                    best_distance = 0
                    print(f"\nSuccess! Found matching WIF: {new_wif}")
                    break
                else:
                    delta = new_distance - current_distance
                    if delta < 0 or random.random() < 2.71828 ** (-delta / temperature):
                        current_key = new_key[:]
                        current_wif = new_wif
                        current_addr = new_addr
                        current_hash160 = new_hash160
                        current_distance = new_distance
                        if delta < 0:
                            print(f"Iteration {iteration + 1}: Distance improved to {current_distance} bits off")
                            print(f"New WIF: {current_wif}")
                            print(f"New Address: {current_addr}")
                            print(f"New Hash160: {current_hash160.hex()}")

                # Update best if improved
                if new_distance < best_distance:
                    best_key = new_key[:]
                    best_distance = new_distance
                    print(f"New best distance: {best_distance} bits off")

                # Cool down temperature
                temperature *= 0.999

                # Progress update
                if (iteration + 1) % 10000 == 0:
                    elapsed = time.time() - start_time
                    print(f"Iteration {iteration + 1}: Current Distance: {current_distance} bits off, Time: {elapsed:.2f}s")

            final_wif = private_key_to_wif(best_key)
            final_addr, final_hash160 = derive_address_and_hash160(best_key)
            final_distance = hamming_distance(final_hash160, target_hash160)
            elapsed = time.time() - start_time
            print("\n--- Analysis Complete ---")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {final_distance} bits off")
            print(f"Time taken: {elapsed:.2f} seconds")
            if final_addr == target_address:
                print("VALIDATION: Exact match found! This WIF controls your target address.")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(max_iterations=100000, tweak_size=6)

Text(value='', description='Target Address:', layout=Layout(width='500px'), placeholder='Enter your target Bit…

Button(description='Submit', style=ButtonStyle())

Output()

In [28]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import hashlib
import base58
import os
import random
from IPython.display import display
import ipywidgets as widgets
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# SECP256k1 curve order
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Function to calculate Hamming distance
def hamming_distance(bytes1, bytes2):
    if len(bytes1) != len(bytes2):
        raise ValueError("Byte strings must be of equal length")
    return sum(bin(b1 ^ b2).count('1') for b1, b2 in zip(bytes1, bytes2))

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes  # Uncompressed, mainnet
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive hash160 and address from private key bytes
def derive_address_and_hash160(private_key_bytes):
    try:
        sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
        vk = sk.verifying_key
        public_key = b'\x04' + vk.to_string()  # Uncompressed
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        hash160 = ripemd160.digest()
        version = b'\x00'
        data = version + hash160
        checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
        address = base58.b58encode(data + checksum).decode('utf-8')
        return address, hash160
    except Exception:
        return None, None

# Function to get target hash160 from address
def address_to_hash160(address):
    decoded = base58.b58decode(address)
    return decoded[1:-4]

# Adaptive search function
def analyze_wif_patterns(max_iterations=500000, tweak_size=5):
    address_input = widgets.Text(
        value='',
        placeholder='Enter your target Bitcoin address',
        description='Target Address:',
        layout={'width': '500px'}
    )
    button = widgets.Button(description="Submit")
    output = widgets.Output()
    display(address_input, button, output)

    def on_button_clicked(b):
        with output:
            output.clear_output()
            target_address = address_input.value.strip()
            if not target_address or not target_address.startswith('1'):
                print("Please enter a valid Bitcoin address starting with '1'.")
                return

            try:
                target_hash160 = address_to_hash160(target_address)
                print(f"Target Address: {target_address}")
                print(f"Target Hash160: {target_hash160.hex()}")
            except Exception as e:
                print(f"Invalid address: {e}")
                return

            # Start with the best key from your last run
            best_wif = "5JxEvR5Z8veRQjBUncvVMtx9pnKz7d26BxEiXZpqHsryF6nsWvS"
            best_key_decoded = base58.b58decode(best_wif)[1:-4]  # Strip version and checksum
            current_key = bytearray(best_key_decoded)
            current_wif = private_key_to_wif(current_key)
            current_addr, current_hash160 = derive_address_and_hash160(current_key)
            current_distance = hamming_distance(current_hash160, target_hash160)
            best_key = current_key[:]
            best_distance = current_distance

            print(f"Starting WIF: {current_wif}")
            print(f"Starting Address: {current_addr}")
            print(f"Starting Distance: {current_distance} bits off")

            start_time = time.time()
            temperature = 10.0  # Lower initial temperature for focus

            for iteration in range(max_iterations):
                # Tweak 1 to tweak_size bytes
                new_key = bytearray(current_key)
                num_tweaks = random.randint(1, tweak_size)
                positions = random.sample(range(32), num_tweaks)
                for pos in positions:
                    new_key[pos] = random.randint(0, 255)

                new_wif = private_key_to_wif(new_key)
                new_addr, new_hash160 = derive_address_and_hash160(new_key)
                if new_hash160 is None:
                    continue
                new_distance = hamming_distance(new_hash160, target_hash160)

                # Prioritize address match, then distance
                if new_addr == target_address:
                    best_key = new_key[:]
                    best_wif = new_wif
                    best_addr = new_addr
                    best_hash160 = new_hash160
                    best_distance = 0
                    print(f"\nSuccess! Found matching WIF: {new_wif}")
                    break
                else:
                    delta = new_distance - current_distance
                    if delta < 0 or random.random() < 2.71828 ** (-delta / temperature):
                        current_key = new_key[:]
                        current_wif = new_wif
                        current_addr = new_addr
                        current_hash160 = new_hash160
                        current_distance = new_distance
                        if delta < 0:
                            print(f"Iteration {iteration + 1}: Distance improved to {current_distance} bits off")
                            print(f"New WIF: {current_wif}")
                            print(f"New Address: {current_addr}")
                            print(f"New Hash160: {current_hash160.hex()}")

                # Update best if improved
                if new_distance < best_distance:
                    best_key = new_key[:]
                    best_distance = new_distance
                    print(f"New best distance: {best_distance} bits off")

                # Cool down temperature faster
                temperature *= 0.99

                # Progress update
                if (iteration + 1) % 50000 == 0:
                    elapsed = time.time() - start_time
                    print(f"Iteration {iteration + 1}: Current Distance: {current_distance} bits off, Time: {elapsed:.2f}s")

            final_wif = private_key_to_wif(best_key)
            final_addr, final_hash160 = derive_address_and_hash160(best_key)
            final_distance = hamming_distance(final_hash160, target_hash160)
            elapsed = time.time() - start_time
            print("\n--- Analysis Complete ---")
            print(f"Best WIF: {final_wif}")
            print(f"Best Address: {final_addr}")
            print(f"Best Hash160: {final_hash160.hex()}")
            print(f"Final Distance: {final_distance} bits off")
            print(f"Time taken: {elapsed:.2f} seconds")
            if final_addr == target_address:
                print("VALIDATION: Exact match found! This WIF controls your target address.")

    button.on_click(on_button_clicked)

# Run the analysis
analyze_wif_patterns(max_iterations=100000, tweak_size=7)

Text(value='', description='Target Address:', layout=Layout(width='500px'), placeholder='Enter your target Bit…

Button(description='Submit', style=ButtonStyle())

Output()

In [31]:
# Install required libraries in Google Colab
!pip install bitcoinlib
!pip install pycryptodome
!pip install ecdsa

import hashlib
import base58
import os
import random
import sys
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160
import time

# SECP256k1 curve order
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Base58 characters from 'K' to 'Z' (excluding I, O)
base58_range = 'KLMNPQRSTUVWXYZ'

# Function to generate a WIF from a private key
def private_key_to_wif(private_key_bytes):
    extended_key = b'\x80' + private_key_bytes
    checksum = hashlib.sha256(hashlib.sha256(extended_key).digest()).digest()[:4]
    wif = base58.b58encode(extended_key + checksum).decode('utf-8')
    return wif

# Function to derive address from private key bytes
def derive_address(private_key_bytes):
    try:
        sk = SigningKey.from_string(private_key_bytes, curve=SECP256k1)
        vk = sk.verifying_key
        public_key = b'\x04' + vk.to_string()  # Uncompressed
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160 = RIPEMD160.new()
        ripemd160.update(sha256_hash)
        hash160 = ripemd160.digest()
        version = b'\x00'
        data = version + hash160
        checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
        address = base58.b58encode(data + checksum).decode('utf-8')
        return address
    except Exception:
        return None

# Scan prefixes for "1F" frequency
def scan_prefixes(target_address, samples_per_prefix=100000):
    print(f"Target Address: {target_address}")
    print("Scanning WIF prefixes from '5K' to '5Z' for '1F' frequency...")

    prefix_counts = {}
    start_time = time.time()

    for char in base58_range:
        prefix = f"5{char}"
        prefix_bytes = base58.b58decode(prefix)
        count_1f = 0

        for _ in range(samples_per_prefix):
            remaining_length = 32 - len(prefix_bytes) + 1  # Adjust for version byte
            random_bytes = prefix_bytes + os.urandom(remaining_length)
            private_key_bytes = random_bytes[:32]
            address = derive_address(private_key_bytes)
            if address and address.startswith("1F"):
                count_1f += 1

        prefix_counts[prefix] = count_1f
        print(f"Prefix {prefix}: {count_1f} '1F' addresses out of {samples_per_prefix}")
        sys.stdout.flush()

    # Rank prefixes
    ranked_prefixes = sorted(prefix_counts.items(), key=lambda x: x[1], reverse=True)
    top_prefix = ranked_prefixes[0][0]
    print(f"\nTop prefix: {top_prefix} with {ranked_prefixes[0][1]} '1F' addresses")
    elapsed = time.time() - start_time
    print(f"Scanning time: {elapsed:.2f} seconds")

    # Search with top prefix
    print(f"\nSearching with prefix {top_prefix} for exact match...")
    return search_with_prefix(target_address, top_prefix)

# Search with a specific prefix
def search_with_prefix(target_address, prefix, max_iterations=5000000):
    prefix_bytes = base58.b58decode(prefix)
    start_time = time.time()

    for iteration in range(max_iterations):
        remaining_length = 32 - len(prefix_bytes) + 1
        random_bytes = prefix_bytes + os.urandom(remaining_length)
        private_key_bytes = random_bytes[:32]
        wif = private_key_to_wif(private_key_bytes)
        address = derive_address(private_key_bytes)

        if address == target_address:
            print(f"\nSuccess! Found matching WIF: {wif}")
            print(f"Address: {address}")
            print(f"Time taken: {time.time() - start_time:.2f} seconds")
            print("VALIDATION: This WIF controls your target address.")
            sys.stdout.flush()
            return wif

        if (iteration + 1) % 500000 == 0:
            elapsed = time.time() - start_time
            print(f"Iteration {iteration + 1}: Still searching with {prefix}... Time: {elapsed:.2f}s")
            sys.stdout.flush()

    elapsed = time.time() - start_time
    print(f"\nNo match found with {prefix} after {max_iterations} iterations.")
    print(f"Time taken: {elapsed:.2f} seconds")
    return None

# Hardcode your target address and run
target_address = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
if target_address and target_address.startswith('1'):
    scan_prefixes(target_address, samples_per_prefix=100000)
else:
    print("Invalid address hardcoded. Please edit the script with a valid Bitcoin address starting with '1'.")

Target Address: 1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF
Scanning WIF prefixes from '5K' to '5Z' for '1F' frequency...
Prefix 5K: 4355 '1F' addresses out of 100000
Prefix 5L: 4333 '1F' addresses out of 100000
Prefix 5M: 4232 '1F' addresses out of 100000
Prefix 5N: 4340 '1F' addresses out of 100000
Prefix 5P: 4403 '1F' addresses out of 100000
Prefix 5Q: 4284 '1F' addresses out of 100000
Prefix 5R: 4398 '1F' addresses out of 100000
Prefix 5S: 4192 '1F' addresses out of 100000
Prefix 5T: 4406 '1F' addresses out of 100000
Prefix 5U: 4387 '1F' addresses out of 100000
Prefix 5V: 4263 '1F' addresses out of 100000
Prefix 5W: 4402 '1F' addresses out of 100000
Prefix 5X: 4443 '1F' addresses out of 100000
Prefix 5Y: 4236 '1F' addresses out of 100000
Prefix 5Z: 4364 '1F' addresses out of 100000

Top prefix: 5X with 4443 '1F' addresses
Scanning time: 1325.22 seconds

Searching with prefix 5X for exact match...
Iteration 500000: Still searching with 5X... Time: 452.39s
Iteration 1000000: Still searching w